In [ ]:
%%time
joined_impacts, df_links = load_and_merge_impact_data(
    unified_path="data/processed/ethiopia_fi_unified_data_enriched.csv",
    links_path="data/processed/impact_links_enriched.csv"
)

In [ ]:
import matplotlib
matplotlib.use('Agg')  # Forces a headless backend (prevents plt.show() from hanging)

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import our production-ready source scripts
from src.impact_modeling import load_and_merge_impact_data, calculate_lagged_impact

# Ensure clean folders exist for artifacts
os.makedirs("data/processed", exist_ok=True)
os.makedirs("reports/figures", exist_ok=True)

# ==========================================
# 1. LOAD AND QUANTIFY DATA SETS
# ==========================================
joined_impacts, df_links = load_and_merge_impact_data(
    unified_path="data/processed/ethiopia_fi_unified_data_enriched.csv",
    links_path="data/processed/impact_links_enriched.csv"
)

print("=== QUANTITATIVE LINKAGE OVERVIEW ===")
# Generate structural mathematical analysis describing counts and mean lag parameters
total_links = len(df_links)
mean_lag = df_links['lag_months'].mean()
max_shock = df_links['impact_magnitude'].max()

print(f"• Total tracked event-to-indicator linkage vectors: {total_links}")
print(f"• Average system transmission lag window: {mean_lag:.2f} months")
print(f"• Maximum target shock magnitude discovered: {max_shock * 100:.1f} percentage points")

print("\n--- Detailed System Linkage Records Matrix ---")
print(joined_impacts[['parent_id', 'related_indicator', 'impact_magnitude', 'lag_months']].to_string())

# ==========================================
# 2. PROJECTION MATRIX AND HEATMAP GENERATION
# ==========================================
association_pivot = df_links.pivot_table(
    index='parent_id', 
    columns='related_indicator', 
    values='impact_magnitude', 
    aggfunc='mean'
).fillna(0.0)

# Verify presence of core required target forecast variables
target_cols = ['ACC_OWNERSHIP', 'ACC_MM_ACCOUNT', 'USG_DIGITAL_PAYMENT']
for col in target_cols:
    if col not in association_pivot.columns:
        association_pivot[col] = 0.0

association_matrix = association_pivot[target_cols]

# Calculate explicit matrix density parameters for quantitative transparency
matrix_density = (association_matrix > 0).mean().mean() * 100
print(f"\n--- Matrix Connectivity Structural Quality ---")
print(f"• Event-Indicator network density index: {matrix_density:.2f}% active connections populated.")

# Render Heatmap with clean layout annotations
plt.figure(figsize=(8, 4.5))
sns.heatmap(association_matrix, annot=True, cmap="Purples", fmt=".2f", cbar=True, linewidths=1, linecolor='gray')
plt.title("Event-Indicator Impact Association Matrix (Max Magnitude)", fontsize=11, fontweight='bold', pad=15)
plt.ylabel("Event Parent ID")
plt.xlabel("Related Inclusion Indicator")
plt.tight_layout()
plt.savefig("reports/figures/task3_association_heatmap.png")
plt.show()

# ==========================================
# 3. MATHEMATICAL TEMPORAL ABSORPTION MODELING
# ==========================================
timeline_years = np.arange(2021, 2026, 0.1)
t_event_telebirr = 2021.36
max_impact_telebirr = 0.15
lag_months_telebirr = 3
lambda_decay_telebirr = 0.05

# Compute vector array output via core engine
telebirr_impact_profile = calculate_lagged_impact(
    t=timeline_years, 
    t_event=t_event_telebirr,    
    max_impact=max_impact_telebirr,      
    lag_months=lag_months_telebirr,          
    lambda_decay=lambda_decay_telebirr      
)

# Calculate explicit quantitative milestones for narrative depth
final_shock_value = telebirr_impact_profile[-1] * 100
saturation_pct = (final_shock_value / (max_impact_telebirr * 100)) * 100

print(f"\n--- Mathematical Absorption Statistics (Telebirr Model Profile) ---")
print(f"• Modeled shock magnitude at end of timeline (2026.0): +{final_shock_value:.2f}% points.")
print(f"• Asymptotic saturation index realized relative to maximum target: {saturation_pct:.2f}% conversion.")

%%time
plt.figure(figsize=(8, 4.5))
# ... your plotting code

# Plot the continuous decay path tracking historical intervals
plt.figure(figsize=(10, 4))
plt.plot(timeline_years, telebirr_impact_profile * 100, color='#117a65', linewidth=2.5)
plt.axvline(x=t_event_telebirr, color='red', linestyle=':', label="Telebirr Launch (May 2021)")
plt.axvline(x=t_event_telebirr + (lag_months_telebirr/12), color='orange', linestyle=':', label="Lag Window Threshold (Aug 2021)")
plt.title("Modeled Temporal Absorption: Telebirr Impact on Digital Payments Usage", fontsize=11, fontweight='bold')
plt.xlabel("Year timeline")
plt.ylabel("Model Shock Contribution (% Points)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig("reports/figures/task3_telebirr_absorption_curve.png")
plt.show()

print("\nExecution complete. Shock analysis artifacts exported to 'reports/figures/'.")